In [2]:
#Step 1: Build the seller-level dataset
import pandas as pd
import sqlite3

# Reload raw tables
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

# Merge order-level data, this time including seller_id and seller_state
df = orders.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
df = df.merge(order_items[['order_id', 'product_id', 'seller_id']], on='order_id', how='left')
df = df.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

df['is_return_proxy'] = ((df['review_score'] <= 2) | (df['order_status'] == 'canceled')).astype(int)

# NOT deduplicating this time — we want one row per seller-item, not per order
print(df.shape)
df[['order_id', 'seller_id', 'product_category_name', 'seller_state', 'is_return_proxy']].head()

(114092, 14)


,order_id,seller_id,product_category_name,seller_state,is_return_proxy
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,utilidades_domesticas,SP,0
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,perfumaria,SP,0
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,automotivo,SP,0
3,949d5b44dbf5de918fe9c16f97b45f8a,66922902710d126a0e7d26b0e3805106,pet_shop,MG,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2c9e548be18521d1c43cde1c582c6de8,papelaria,SP,0


In [3]:
#Step 2: Load into SQLite
# Create (or connect to) a local SQLite database file
conn = sqlite3.connect('../data/processed/returnflow.db')

# Load the dataframe into it as a table called "order_items_full"
df.to_sql('order_items_full', conn, if_exists='replace', index=False)

# Quick check: run a simple SQL query to confirm it loaded correctly
check = pd.read_sql("SELECT COUNT(*) as total_rows FROM order_items_full", conn)
print(check)

   total_rows
0      114092


In [4]:
#Step 3: Write the SQL — seller return rate vs. category average (window function)
query = """
WITH seller_stats AS (
    SELECT
        seller_id,
        product_category_name,
        COUNT(*) AS total_items,
        AVG(is_return_proxy) AS seller_return_rate
    FROM order_items_full
    WHERE product_category_name IS NOT NULL
    GROUP BY seller_id, product_category_name
)
SELECT
    seller_id,
    product_category_name,
    total_items,
    ROUND(seller_return_rate, 4) AS seller_return_rate,
    ROUND(AVG(seller_return_rate) OVER (PARTITION BY product_category_name), 4) AS category_avg_return_rate,
    ROUND(seller_return_rate - AVG(seller_return_rate) OVER (PARTITION BY product_category_name), 4) AS diff_from_category_avg
FROM seller_stats
WHERE total_items >= 5
ORDER BY diff_from_category_avg DESC
LIMIT 20
"""

result = pd.read_sql(query, conn)
result

,seller_id,product_category_name,total_items,seller_return_rate,category_avg_return_rate,diff_from_category_avg
0,d4a5e99e0dd915df64ba55a7fbd583fd,beleza_saude,5,1.0000,0.1269,0.8731
1,8444e55c1f13cd5c179851e5ca5ebd00,fashion_bolsas_e_acessorios,5,1.0000,0.1274,0.8726
2,0725b8c0f3f906e58f70cbe76b7c748c,esporte_lazer,6,1.0000,0.1471,0.8529
3,a23266650e7c84bb93fbbba502137478,ferramentas_jardim,5,1.0000,0.1477,0.8523
4,73a63f72308aa20a46f4b1632018f196,eletronicos,5,1.0000,0.1483,0.8517
5,ec2e006556300a79a5a91e4876ab3a56,construcao_ferramentas_construcao,7,1.0000,0.1511,0.8489
6,be3b4b0f050a6aa1b2d901c4b77e979f,utilidades_domesticas,6,1.0000,0.1538,0.8462
7,bfd938b22bc99bce1ae60dc602889f52,utilidades_domesticas,6,1.0000,0.1538,0.8462
8,a0e19590a0923cdd0614ea9427713ced,artes,7,1.0000,0.1691,0.8309
9,90d4125885ab6c86e8820a722be71974,consoles_games,5,1.0000,0.1879,0.8121


In [5]:
query_refined = """
WITH seller_stats AS (
    SELECT
        seller_id,
        product_category_name,
        COUNT(*) AS total_items,
        AVG(is_return_proxy) AS seller_return_rate
    FROM order_items_full
    WHERE product_category_name IS NOT NULL
    GROUP BY seller_id, product_category_name
)
SELECT
    seller_id,
    product_category_name,
    total_items,
    ROUND(seller_return_rate, 4) AS seller_return_rate,
    ROUND(AVG(seller_return_rate) OVER (PARTITION BY product_category_name), 4) AS category_avg_return_rate,
    ROUND(seller_return_rate - AVG(seller_return_rate) OVER (PARTITION BY product_category_name), 4) AS diff_from_category_avg
FROM seller_stats
WHERE total_items >= 15
ORDER BY diff_from_category_avg DESC
LIMIT 15
"""

result_refined = pd.read_sql(query_refined, conn)
result_refined

,seller_id,product_category_name,total_items,seller_return_rate,category_avg_return_rate,diff_from_category_avg
0,4342d4b2ba6b161468c63a7e7cfce593,relogios_presentes,20,0.9000,0.1780,0.7220
1,b1b3948701c5c72445495bd161b83a4c,automotivo,18,0.7778,0.1559,0.6218
2,1ca7077d890b907f89be8c954a02686a,brinquedos,25,0.7600,0.1428,0.6172
3,710e3548e02bc1d2831dfc4f1b5b14d4,informatica_acessorios,30,0.7667,0.1880,0.5787
4,2709af9587499e95e803a6498a5a56e9,beleza_saude,37,0.6216,0.1332,0.4885
5,bb135baca94c82fcb731335ad5b04a03,moveis_decoracao,29,0.6552,0.1898,0.4654
6,d12c926d74ceff0a90a21184466ce161,perfumaria,20,0.6000,0.1601,0.4399
7,5bc55dbe2f12b6af6d83ed46023e0dc8,esporte_lazer,19,0.5789,0.1527,0.4262
8,e250d617a0ad591ba9bd663e584a895d,moveis_decoracao,20,0.6000,0.1898,0.4102
9,c6381d2d013342748761e906d45aff76,utilidades_domesticas,25,0.5600,0.1512,0.4088


In [6]:
query_ranked = """
WITH seller_stats AS (
    SELECT
        seller_id,
        product_category_name,
        COUNT(*) AS total_items,
        AVG(is_return_proxy) AS seller_return_rate
    FROM order_items_full
    WHERE product_category_name IS NOT NULL
    GROUP BY seller_id, product_category_name
    HAVING COUNT(*) >= 15
)
SELECT
    seller_id,
    product_category_name,
    total_items,
    ROUND(seller_return_rate, 4) AS seller_return_rate,
    RANK() OVER (PARTITION BY product_category_name ORDER BY seller_return_rate DESC) AS rank_within_category
FROM seller_stats
ORDER BY product_category_name, rank_within_category
LIMIT 30
"""

result_ranked = pd.read_sql(query_ranked, conn)
result_ranked

,seller_id,product_category_name,total_items,seller_return_rate,rank_within_category
0,e59aa562b9f8076dd550fcddf0e73491,agro_industria_e_comercio,84,0.1071,1
1,6481e96574816ead57975da2c0f6d80d,agro_industria_e_comercio,15,0.0667,2
2,9b013e03b2ab786505a1d3b5c0756754,alimentos,27,0.3333,1
3,e9779976487b77c6d4ac45f75ec7afe9,alimentos,26,0.2308,2
4,16090f2ca825584b5a147ab24aa30c86,alimentos,117,0.1453,3
5,8d79c8a04e42d722a75097ce5cbcf2ef,alimentos,28,0.1071,4
6,cbd996ad3c1b7dc71fd0e5f5df9087e2,alimentos,99,0.0808,5
7,a5a1bfcf728ab0e19182959cf0771ee4,alimentos,18,0.0556,6
8,d13e50eaa47b4cbe9eb81465865d8cfc,alimentos,58,0.0345,7
9,5011f0d93373a4c5753adf58ca77af8d,alimentos_bebidas,17,0.2353,1
